# ISM Sweep on Colab (GPU-accelerated Model B)

Runs the full 514-sample In-silico Saturation Mutagenesis sweep (issue #11) using Colab's free GPU to accelerate Model B's NT transformer embedding pass - the actual bottleneck (~0.29s/sequence on CPU, ~4.5h for the full 56,026-sequence sweep locally). RNAfold/RNAduplex physical-feature computation is already batched and fast (~4 min total for the full sweep) and runs on CPU regardless (ViennaRNA has no GPU path), so it isn't the target here.

**One-time setup**: upload these files/folders to a folder in your Google Drive: `best_model_fold1.pth`, `NT_sacas9_fintuned_model/` (the whole checkpoint directory), `hybrid_xgb_model.json`, `test_metadata.csv`. Set `DRIVE_DIR` below to that folder's path.

**Output**: `ism_delta.npz` (keys `ism_delta_model_a`, `ism_delta_model_b`, each shape `(514, 36, 3)`) - small (well under 1MB), downloaded automatically at the end and also kept on Drive as a backup. Copy it back into your local `xai_test/` directory.


## 1. Mount Google Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Point at your uploaded model assets

Edit `DRIVE_DIR` to the Drive folder holding `best_model_fold1.pth`, `NT_sacas9_fintuned_model/`, `hybrid_xgb_model.json`, `test_metadata.csv`.


In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/xai_test"  # <-- EDIT ME

required = [
    "best_model_fold1.pth",
    "NT_sacas9_fintuned_model",
    "hybrid_xgb_model.json",
    "test_metadata.csv",
]
missing = [
    name for name in required if not os.path.exists(os.path.join(DRIVE_DIR, name))
]
if missing:
    raise FileNotFoundError(
        f"Missing in {DRIVE_DIR}: {missing}. Upload these to Drive first, or fix DRIVE_DIR."
    )

os.chdir(DRIVE_DIR)
print("Working directory:", os.getcwd())

## 3. Install ViennaRNA CLI (RNAfold / RNAduplex) and Python deps

If `apt-get install vienna-rna` fails (package not found on this Colab image), say so and I'll add a conda-forge/bioconda fallback - don't silently skip this step, `Model_B_Predictor` will fail without both binaries on `PATH`.


In [ ]:
!apt-get -qq update && apt-get -qq install -y vienna-rna
!which RNAfold && which RNAduplex

In [ ]:
!pip install -q transformers biopython xgboost

## 4. Write the predictor + ISM sweep modules

Mirrors the current committed `xai_test/` repo exactly (embedded at notebook-build time, not fetched at runtime).


In [ ]:
%%writefile model_a_wrapper.py
import torch
import torch.nn as nn
import numpy as np
from typing import List

class OneHotAdapter(nn.Module):
    def __init__(self, input_dim: int = 4, hidden_dim: int = 128):
        super().__init__()
        # 가중치 형태 [128, 4]에 맞게 nn.Linear로 변경
        self.proj = nn.Linear(input_dim, hidden_dim)

    def forward(self, x):
        # x shape: [Batch, 36, 4]
        x = self.proj(x)          # -> [Batch, 36, 128]
        return x.transpose(1, 2)  # CNN 백본 입력에 맞게 변경 -> [Batch, 128, 36]

class CNN_RNN_Backbone(nn.Module):
    def __init__(
        self,
        input_dim: int = 128,
        conv_channels: int = 64,
        kernel_size: int = 3,
        lstm_hidden: int = 128,
        lstm_layers: int = 1,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.input_dim = input_dim
        self.conv = nn.Conv1d(input_dim, conv_channels, kernel_size, padding=kernel_size // 2)
        self.act = nn.ReLU()
        self.lstm = nn.LSTM(
            input_size=conv_channels,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden),
            nn.ReLU(),
            nn.Linear(lstm_hidden, 1),
        )

    def forward(self, x):
        # x shape: [Batch, 128, 36]
        x = self.act(self.conv(x))
        x = x.transpose(1, 2)  # [Batch, 36, 64]
        out, _ = self.lstm(x)  # out shape: [Batch, 36, 128]
        
        # 마지막 시점(sequence)의 hidden state를 추출하여 분류기 통과
        out = self.head(out[:, -1, :])  
        return out

class IntegratedPredictor(nn.Module):
    def __init__(self, adapter: nn.Module, backbone: nn.Module):
        super().__init__()
        self.adapter = adapter
        self.backbone = backbone

    def forward(self, x):
        x = self.adapter(x)
        x = self.backbone(x)
        return x.squeeze(-1) # 1D Array 변환

class Model_A_Predictor:
    def __init__(self, weight_path: str = None, dropout_rate: float = 0.0):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        # 원본 구조와 동일하게 어댑터와 백본 조립
        adapter = OneHotAdapter(input_dim=4, hidden_dim=128)
        backbone = CNN_RNN_Backbone(input_dim=128, lstm_hidden=128, dropout=dropout_rate)
        self.model = IntegratedPredictor(adapter=adapter, backbone=backbone).to(self.device)
        
        # 가중치 로드
        if weight_path:
            self.model.load_state_dict(torch.load(weight_path, map_location=self.device))
        self.model.eval()

        self.nuc_map = {
            'A': [1.0, 0.0, 0.0, 0.0],
            'C': [0.0, 1.0, 0.0, 0.0],
            'G': [0.0, 0.0, 1.0, 0.0],
            'T': [0.0, 0.0, 0.0, 1.0]
        }

    def _encode_one_hot(self, sequences: List[str]) -> torch.Tensor:
        encoded_list = []
        for seq in sequences:
            encoded = [self.nuc_map.get(n.upper(), [0.0, 0.0, 0.0, 0.0]) for n in seq]
            encoded_list.append(encoded)
        tensor = torch.tensor(encoded_list, dtype=torch.float32)
        return tensor # shape: [Batch, 36, 4]

    def predict(self, sequences: List[str]) -> np.ndarray:
        x = self._encode_one_hot(sequences).to(self.device)
        with torch.no_grad():
            preds = self.model(x).cpu().numpy()
        return preds

In [ ]:
%%writefile model_b_wrapper.py
import os
import pickle
import re
import subprocess

import joblib
import numpy as np
import torch
import xgboost as xgb
from Bio.SeqUtils import MeltingTemp as mt
from Bio.SeqUtils import gc_fraction
from transformers import AutoModelForMaskedLM, AutoTokenizer


class Model_B_Predictor:
    def __init__(self, nt_model_dir: str, xgb_model_path: str):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"Loading NT backbone from {nt_model_dir}...")
        self.tokenizer = AutoTokenizer.from_pretrained(nt_model_dir)
        self.nt_model = AutoModelForMaskedLM.from_pretrained(nt_model_dir).to(
            self.device
        )
        self.nt_model.eval()

        print(f"Loading XGBoost model from {xgb_model_path}...")
        self.xgb_model = self._load_xgb_model(xgb_model_path)

    def _load_xgb_model(self, path: str):
        if not os.path.exists(path):
            raise FileNotFoundError(f"XGBoost model file not found: {path}")

        try:
            with open(path, "rb") as f:
                return pickle.load(f)
        except Exception:  # noqa: BLE001 - pre-existing multi-format load fallback
            try:
                return joblib.load(path)
            except Exception:  # noqa: BLE001 - pre-existing multi-format load fallback
                model = xgb.XGBRegressor()
                model.load_model(path)
                return model

    def _extract_nt_embeddings(self, sequences: list[str]) -> np.ndarray:
        inputs = self.tokenizer(
            sequences, return_tensors="pt", padding=True, truncation=True
        ).to(self.device)
        with torch.no_grad():
            outputs = self.nt_model(**inputs, output_hidden_states=True)
            last_hidden_state = outputs.hidden_states[-1]

            batch_size, seq_len, hidden_dim = last_hidden_state.shape
            assert seq_len == 7, f"Expected 7 tokens, got {seq_len}"
            assert hidden_dim == 1280, f"Expected 1280 hidden dim, got {hidden_dim}"

            flattened_embeddings = last_hidden_state.view(batch_size, -1).cpu().numpy()
        return flattened_embeddings

    def _get_guide_rna(self, dna_seq: str) -> str:
        complement_map = str.maketrans("ATCGatcg", "UAGCuagc")
        return dna_seq.translate(complement_map)[::-1]

    # RNAfold/RNAduplex are external processes: each subprocess.run() call pays
    # OS process-spawn overhead regardless of how little work it does. Both CLIs
    # accept many records over a single stdin stream, so batching sequences into
    # one call (chunked, to keep any single call's stdin/stdout bounded) turns an
    # O(n) process-spawn cost into O(n / batch_size) - the dominant cost at
    # ISM-sweep scale (tens of thousands of mutants). Deliberate scope override of
    # issue #9's "RNAfold/RNAduplex call batching" Out-of-Scope note, requested
    # directly by the user; verified to reproduce the exact same per-sequence
    # energy values as the prior one-process-per-sequence implementation.
    PHYSICAL_FEATURE_BATCH_SIZE = 500

    @staticmethod
    def _chunked(items: list, size: int):
        for i in range(0, len(items), size):
            yield items[i : i + size]

    def _calc_mfe_batch(self, seqs: list[str]) -> list[float]:
        energies: list[float] = []
        for chunk in self._chunked(seqs, self.PHYSICAL_FEATURE_BATCH_SIZE):
            # RNAfold CLI 호출 (--noPS: 이미지 생성 방지)
            proc = subprocess.run(
                ["RNAfold", "--noPS"],
                input="".join(f"{seq}\n" for seq in chunk),
                capture_output=True,
                text=True,
                check=True,
            )
            # Each record prints "<sequence>\n<dot-bracket structure> (<energy>)\n".
            # The structure diagram's own parentheses never have a digit directly
            # inside them, so this regex matches exactly one energy per record,
            # in input order. 출력 예시: ... ( -4.20)
            matches = re.findall(r"\(\s*(-?\d+(?:\.\d+)?)\)", proc.stdout)
            assert len(matches) == len(chunk), (
                f"RNAfold batch output mismatch: expected {len(chunk)} results, got {len(matches)}"
            )
            energies.extend(float(m) for m in matches)
        return energies

    def _calc_duplex_dg_batch(self, pairs: list[tuple[str, str]]) -> list[float]:
        energies: list[float] = []
        for chunk in self._chunked(pairs, self.PHYSICAL_FEATURE_BATCH_SIZE):
            # RNAduplex CLI 호출
            proc = subprocess.run(
                ["RNAduplex"],
                input="".join(f"{guide}\n{target}\n" for guide, target in chunk),
                capture_output=True,
                text=True,
                check=True,
            )
            # 출력 예시: ... :  1,36 :   1,36 (-32.50)
            matches = re.findall(r"\(\s*(-?\d+(?:\.\d+)?)\)", proc.stdout)
            assert len(matches) == len(chunk), (
                f"RNAduplex batch output mismatch: expected {len(chunk)} results, got {len(matches)}"
            )
            energies.extend(float(m) for m in matches)
        return energies

    def _compute_physical_features(self, sequences: list[str]) -> np.ndarray:
        # 1. MFE (RNAfold.exe), batched across all sequences
        mfe_values = self._calc_mfe_batch(sequences)

        # 2. Duplex ΔG (RNAduplex.exe), batched across all sequences
        guide_rnas = [self._get_guide_rna(seq) for seq in sequences]
        dg_values = self._calc_duplex_dg_batch(list(zip(guide_rnas, sequences)))

        features = []
        for seq, mfe, dg in zip(sequences, mfe_values, dg_values):
            # 3. Tm (must match training-time nn_table=DNA_NN4 in nt_feature_stacking.py)
            tm = mt.Tm_NN(seq, nn_table=mt.DNA_NN4)

            # 4. GC Content
            gc = gc_fraction(seq) * 100.0

            features.append([mfe, dg, tm, gc])
        return np.array(features, dtype=np.float32)

    def predict(self, sequences: list[str]) -> np.ndarray:
        embeddings = self._extract_nt_embeddings(sequences)
        physical_feats = self._compute_physical_features(sequences)

        final_features = np.hstack([embeddings, physical_feats])
        assert final_features.shape[1] == 8964, (
            f"Dimension mismatch: Expected 8964, got {final_features.shape[1]}"
        )

        preds = self.xgb_model.predict(final_features)
        return np.array(preds).flatten()


In [ ]:
%%writefile ism_sweep.py
"""In-silico saturation mutagenesis (ISM) sweep (Step 3, see CONTEXT.md / plan_realize.md).

Generates every position x alternative-base substitution for a set of 36bp
Testset sequences, scores wild-type and mutants through a predictor's own
`predict()`, and reduces the result to a relative delta-score sensitivity
array. Works against any predictor exposing `predict(sequences) -> array`
(duck-typed to `Model_A_Predictor` / `Model_B_Predictor`), so Model B's
per-mutant physical-feature recomputation (MFE, dG, Tm, GC) happens for free:
every mutant is dispatched as a full sequence, never a cached wild-type value
or a bare delta.
"""

from collections.abc import Sequence

import numpy as np

BASES = ("A", "C", "G", "T")

# Relative delta-Score is undefined (unbounded) as Score_WT -> 0; scores at or
# below this magnitude are masked to NaN instead of producing a spurious ratio.
RELATIVE_DELTA_NAN_THRESHOLD = 1e-4


def alt_bases_for(wt_base: str) -> list[str]:
    """The 3 non-wild-type bases, in fixed lexicographic order."""
    wt_base = wt_base.upper()
    if wt_base not in BASES:
        raise ValueError(f"Not a valid base: {wt_base!r}")
    return sorted(set(BASES) - {wt_base})


def generate_ism_mutants(sequence: str) -> list[str]:
    """All position x alternative-base substitutions for `sequence`.

    Ordered position-major (0..len-1), alternative-base lexicographic-minor,
    matching the `ism_delta` array's (position, alternative_base) axes.
    """
    sequence = sequence.upper()
    mutants = []
    for pos, wt_base in enumerate(sequence):
        for alt_base in alt_bases_for(wt_base):
            mutants.append(sequence[:pos] + alt_base + sequence[pos + 1 :])
    return mutants


def compute_ism_delta(
    sequences: Sequence[str],
    predictor,
    nan_threshold: float = RELATIVE_DELTA_NAN_THRESHOLD,
) -> np.ndarray:
    """Run the full ISM sweep for one predictor.

    Returns `ism_delta`, shape `(len(sequences), seq_len, 3)`: the relative
    score delta `(Score_mutant - Score_WT) / |Score_WT|` per position x
    lexicographic alternative base, NaN-masked wherever `|Score_WT| <=
    nan_threshold`.
    """
    sequences = list(sequences)
    seq_len = len(sequences[0])
    if any(len(seq) != seq_len for seq in sequences):
        raise ValueError("All sequences must share the same length for the ISM sweep.")

    wt_scores = np.asarray(predictor.predict(sequences), dtype=np.float64)

    mutants: list[str] = []
    for sequence in sequences:
        mutants.extend(generate_ism_mutants(sequence))

    mutant_scores = np.asarray(predictor.predict(mutants), dtype=np.float64)
    mutant_scores = mutant_scores.reshape(len(sequences), seq_len, 3)

    wt_scores_broadcast = wt_scores[:, None, None]
    with np.errstate(divide="ignore", invalid="ignore"):
        ism_delta = (mutant_scores - wt_scores_broadcast) / np.abs(wt_scores_broadcast)

    ism_delta[np.abs(wt_scores) <= nan_threshold] = np.nan

    return ism_delta


def run_ism_sweep(
    sequences: Sequence[str], predictor_a, predictor_b
) -> dict[str, np.ndarray]:
    """Run the ISM sweep for both models over the same Testset sequences."""
    return {
        "ism_delta_model_a": compute_ism_delta(sequences, predictor_a),
        "ism_delta_model_b": compute_ism_delta(sequences, predictor_b),
    }


## 5. Confirm GPU is available


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected - Runtime > Change runtime type > GPU")

## 6. Load the Testset


In [ ]:
import pandas as pd

meta_df = pd.read_csv("test_metadata.csv")
sequences = meta_df["sequence"].tolist()
print(f"Loaded {len(sequences)} testset sequences")

## 7. Run the full ISM sweep


In [ ]:
from model_a_wrapper import Model_A_Predictor
from model_b_wrapper import Model_B_Predictor

predictor_a = Model_A_Predictor(weight_path="best_model_fold1.pth")
predictor_b = Model_B_Predictor(
    nt_model_dir="./NT_sacas9_fintuned_model", xgb_model_path="hybrid_xgb_model.json"
)

In [ ]:
import time

import numpy as np
from ism_sweep import run_ism_sweep

t0 = time.time()
result = run_ism_sweep(sequences, predictor_a, predictor_b)
elapsed = time.time() - t0
print(f"ISM sweep completed in {elapsed / 60:.1f} min")
for name, arr in result.items():
    nan_samples = np.isnan(arr).any(axis=(1, 2)).sum()
    print(f" - {name}: shape={arr.shape}, NaN-masked samples={nan_samples}")

## 8. Save and download the result


In [ ]:
import numpy as np

np.savez("ism_delta.npz", **result)
# Also keep a copy on Drive in case the runtime disconnects before you download.
np.savez(os.path.join(DRIVE_DIR, "ism_delta.npz"), **result)

from google.colab import files

files.download("ism_delta.npz")

Copy the downloaded `ism_delta.npz` into your local `xai_test/` directory - it has the same two keys `step3_ism.py` produces locally.
